# odmlib v0.2.0 Error Handling

odmlib v0.2.0 introduces a structured exception hierarchy that replaces the generic `ValueError` and `TypeError` exceptions used in earlier versions. Every odmlib-specific error now inherits from `OdmlibError`, and each exception carries rich context attributes — element paths, hints, attribute names — that make diagnosing problems straightforward.

This notebook covers:

1. **Exception hierarchy** — the new error types and how they relate to each other
2. **Structured error attributes** — accessing context, hints, and element paths on exceptions
3. **Type validation errors** — what happens when you assign the wrong type to an attribute
4. **Required attribute errors** — catching missing required attributes
5. **Element order errors** — detecting and fixing out-of-order child elements
6. **OID validation errors** — duplicate OIDs and broken references
7. **Conformance errors** — Cerberus-based metadata conformance checking
8. **Collect-all-errors mode** — gathering every validation problem in a single pass
9. **Warnings** — non-fatal issues and deprecation notices
10. **Backward compatibility** — how existing `except ValueError` code still works

## Setup

Import the odmlib model classes and the new exception types. All exceptions are exported from the top-level `odmlib` package.

In [20]:
from datetime import datetime, timezone
import warnings

import odmlib.define_2_1.model as DEFINE
import odmlib.odm_1_3_2.model as ODM

from odmlib import (
    OdmlibError,
    OdmlibValidationError,
    OdmlibRequiredAttributeError,
    OdmlibTypeError,
    OdmlibOIDError,
    OdmlibConformanceError,
    OdmlibElementOrderError,
    OdmlibWarning,
    OdmlibDeprecationWarning,
    ErrorCollector,
    create_oid_checker,
)

## 1. The Exception Hierarchy

All odmlib exceptions inherit from a single base class, `OdmlibError`. This makes it easy to catch any odmlib problem with a single `except` clause, or to catch specific subtypes when you need more control.

```
Exception
├── OdmlibError                        # base for all odmlib exceptions
│   ├── OdmlibValidationError          # validation failures (also inherits ValueError)
│   │   ├── OdmlibRequiredAttributeError   # missing required attribute
│   │   ├── OdmlibOIDError                 # OID uniqueness / ref-def integrity
│   │   ├── OdmlibConformanceError         # Cerberus conformance failures
│   │   └── OdmlibElementOrderError        # child elements out of spec order
│   ├── OdmlibTypeError               # wrong type for an attribute (also inherits TypeError)
│   ├── OdmlibParsingError            # XML/JSON parsing failures
│   │   └── OdmlibLoaderStateError    # loader called before document opened
│   ├── OdmlibSerializationError      # cannot write model to XML/JSON
│   └── OdmlibNamespaceError          # namespace registration/lookup failures
│
UserWarning
├── OdmlibWarning                     # base for non-fatal warnings
│   ├── OdmlibDeprecationWarning      # deprecated features (also DeprecationWarning)
│   └── OdmlibInteroperabilityWarning # valid but risky constructs
```

Let's verify the inheritance relationships:

In [21]:
# Every odmlib exception is an OdmlibError
print(f"OdmlibValidationError is OdmlibError: {issubclass(OdmlibValidationError, OdmlibError)}")
print(f"OdmlibTypeError is OdmlibError:       {issubclass(OdmlibTypeError, OdmlibError)}")
print(f"OdmlibOIDError is OdmlibError:        {issubclass(OdmlibOIDError, OdmlibError)}")

# Specific subtypes
print(f"\nOdmlibOIDError is OdmlibValidationError:        {issubclass(OdmlibOIDError, OdmlibValidationError)}")
print(f"OdmlibConformanceError is OdmlibValidationError: {issubclass(OdmlibConformanceError, OdmlibValidationError)}")
print(f"OdmlibElementOrderError is OdmlibValidationError: {issubclass(OdmlibElementOrderError, OdmlibValidationError)}")

# Backward compatibility: validation errors are still ValueErrors
print(f"\nOdmlibValidationError is ValueError: {issubclass(OdmlibValidationError, ValueError)}")
print(f"OdmlibTypeError is TypeError:         {issubclass(OdmlibTypeError, TypeError)}")

OdmlibValidationError is OdmlibError: True
OdmlibTypeError is OdmlibError:       True
OdmlibOIDError is OdmlibError:        True

OdmlibOIDError is OdmlibValidationError:        True
OdmlibConformanceError is OdmlibValidationError: True
OdmlibElementOrderError is OdmlibValidationError: True

OdmlibValidationError is ValueError: True
OdmlibTypeError is TypeError:         True


## 2. Structured Error Attributes

Unlike plain `ValueError` messages, the new exceptions carry structured metadata that you can inspect programmatically. Every `OdmlibValidationError` and `OdmlibTypeError` can include:

- **`element_path`** — a human-readable path from the root to the failing element
- **`hint`** — a suggestion for how to fix the problem
- **`attribute`** — the specific attribute that failed
- **`element_type`** — the class name of the element
- **`actual_value`** — the value that was provided

These attributes are included in the formatted error message and are also available as Python attributes on the exception object for programmatic access.

In [22]:
# Trigger a type error by assigning an integer where a string is expected
try:
    item = ODM.ItemDef(OID="IT.STUDYID", Name="STUDYID", DataType="text", Length=200)
    item.Origin = 12345  # Origin expects a string, not an integer
except OdmlibTypeError as e:
    print("=== Formatted error message ===")
    print(e)
    print("\n=== Structured attributes ===")
    print(f"  attribute:     {e.attribute}")
    print(f"  expected_type: {e.expected_type}")
    print(f"  actual_value:  {e.actual_value}")
    print(f"  hint:          {e.hint}")

=== Formatted error message ===
Expected type <class 'str'> for Origin with value 12345
  Hint: Provide a value of type str

=== Structured attributes ===
  attribute:     Origin
  expected_type: <class 'str'>
  actual_value:  12345
  hint:          Provide a value of type str


The structured attributes let you build tooling on top of odmlib — for example, an IDE plugin that highlights the specific attribute, or a batch validator that groups errors by type.

## 3. Type Validation Errors (`OdmlibTypeError`)

`OdmlibTypeError` is raised immediately when you assign a value of the wrong type to an ODM attribute. odmlib validates at assignment time, so you find out about type mismatches right away — not later when you try to serialize.

Here are several examples of type errors you might encounter.

In [23]:
# Example 1: Wrong type for a string attribute
try:
    study = ODM.Study(OID=999)  # OID must be a string
except OdmlibTypeError as e:
    print(f"String type error: {e.attribute} — got {e.actual_value!r}")
    print(f"  Hint: {e.hint}")

String type error: OID — got 999
  Hint: Provide a value of type str


In [24]:
# Example 2: Non-integer where an integer is expected
try:
    item_ref = ODM.ItemRef(ItemOID="IT.STUDYID", OrderNumber="not_a_number", Mandatory="Yes")
except OdmlibTypeError as e:
    print(f"Integer conversion error: {e.attribute} — got {e.actual_value!r}")
    print(f"  Hint: {e.hint}")

Integer conversion error: OrderNumber — got 'not_a_number'
  Hint: Provide a string that represents a valid integer, e.g., '42'


In [25]:
# Example 3: Invalid enumerated value
try:
    item = ODM.ItemDef(OID="IT.STUDYID", Name="STUDYID", DataType="invalid_type", Length=200)
except OdmlibTypeError as e:
    print(f"Enumeration error: {e.attribute} — got {e.actual_value!r}")
    print(f"  Hint: {e.hint}")

Enumeration error: DataType — got 'invalid_type'
  Hint: Value must be one of: text, integer, float, date, time, datetime, string, boolean, double, hexBinary, base64Binary, hexFloat, base64Float, partialDate, partialTime, partialDatetime, durationDatetime, intervalDatetime, incompleteDatetime, incompleteDate, incompleteTime, URI


## 4. Required Attribute Errors (`OdmlibRequiredAttributeError`)

If you try to access a required attribute that was not provided during construction, odmlib raises `OdmlibRequiredAttributeError`. This is a subclass of `OdmlibValidationError`, so it carries the same structured attributes.

In [26]:
# Create an ItemDef without the required DataType attribute, then try to access it
item = ODM.ItemDef.__new__(ODM.ItemDef)  # bypass __init__ to skip validation
item.__dict__["OID"] = "IT.TEST"
item.__dict__["Name"] = "TEST"
# DataType was never set

try:
    _ = item.DataType  # accessing the missing required attribute
except OdmlibRequiredAttributeError as e:
    print(f"Missing attribute: {e.attribute}")
    print(f"Element type: {e.element_type}")
    print(f"Hint: {e.hint}")

Missing attribute: DataType
Element type: ItemDef
Hint: Attribute 'DataType' is required when constructing ItemDef


## 5. Element Order Errors (`OdmlibElementOrderError`)

The ODM specification requires child elements to appear in a specific order. The `verify_order()` method checks this recursively and raises `OdmlibElementOrderError` if any element has children out of order.

odmlib also provides `reorder_object()` to fix ordering automatically.

In [27]:
# Build a small Define-XML document with elements in the wrong order
current_datetime = datetime.now(timezone.utc).isoformat()

odm = DEFINE.ODM(
    FileOID="DEF.DEMO",
    FileType="Snapshot",
    CreationDateTime=current_datetime,
    ODMVersion="1.3.2",
    Context="Submission",
    Originator="odmlib",
    SourceSystem="odmlib",
    SourceSystemVersion="0.2.0",
)

study = DEFINE.Study(OID="ST.DEMO")
study.GlobalVariables = DEFINE.GlobalVariables()
study.GlobalVariables.StudyName = ODM.StudyName(_content="Demo Study")
study.GlobalVariables.StudyDescription = ODM.StudyDescription(_content="Error handling demo")
study.GlobalVariables.ProtocolName = ODM.ProtocolName(_content="DEMO-001")

mdv = DEFINE.MetaDataVersion(OID="MDV.DEMO", Name="Demo MDV", DefineVersion="2.1.0")
study.MetaDataVersion = mdv
odm.Study = study

# Add an ItemDef before an ItemGroupDef — this is the wrong order per the ODM spec
item = ODM.ItemDef(OID="IT.STUDYID", Name="STUDYID", DataType="text", Length=200)
mdv.ItemDef.append(item)

igd = DEFINE.ItemGroupDef(
    OID="IG.DM", Name="DM", Repeating="No",
    IsReferenceData="No", SASDatasetName="DM",
    Structure="One record per subject", Purpose="Tabulation",
)
igd.ItemRef.append(ODM.ItemRef(ItemOID="IT.STUDYID", OrderNumber=1, Mandatory="Yes"))
mdv.ItemGroupDef.append(igd)

print("Document built with elements in the wrong order.")

Document built with elements in the wrong order.


In [28]:
# verify_order() detects the problem
try:
    mdv.verify_order()
except OdmlibElementOrderError as e:
    print(f"Order error in: {e.element_type}")
    print(f"Hint: {e.hint}")
    print(f"\nFull message:\n{e}")

Order error in: MetaDataVersion
Hint: Use reorder_object() to fix element ordering automatically

Full message:
The order of elements in MetaDataVersion should be Standards, AnnotatedCRF, SupplementalDoc, ValueListDef, WhereClauseDef, ItemGroupDef, ItemDef, CodeList, MethodDef, CommentDef, leaf
  Hint: Use reorder_object() to fix element ordering automatically


In [29]:
# Fix the ordering automatically with reorder_object()
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    mdv.reorder_object()
    if caught:
        print(f"Warning issued: {caught[0].message}")

# Now verify_order() passes
result = mdv.verify_order()
print(f"\nOrder verified: {result}")

Warning issued: MetaDataVersion elements are being reordered to match the ODM specification. Use verify_order() before reorder_object() to understand the ordering issue.

Order verified: True


## 6. OID Validation Errors (`OdmlibOIDError`)

OID validation checks two things:
1. **Uniqueness** — each OID definition must be unique within its scope
2. **Referential integrity** — every OID reference must point to an existing definition

odmlib v0.2.0 introduces `create_oid_checker()`, which dynamically introspects the model to build the ref/def mappings. This replaces the manually-coded OIDRef classes from earlier versions.

In [30]:
# Create a checker for the Define-XML 2.1 model
checker = create_oid_checker("define_2_1")

# Add a broken reference — this ItemRef points to an OID that does not exist
igd2 = DEFINE.ItemGroupDef(
    OID="IG.VS", Name="VS", Repeating="Yes",
    IsReferenceData="No", SASDatasetName="VS",
    Structure="One record per subject per visit per test", Purpose="Tabulation",
)
igd2.ItemRef.append(ODM.ItemRef(ItemOID="IT.NONEXISTENT", OrderNumber=1, Mandatory="Yes"))
mdv.ItemGroupDef.append(igd2)

try:
    odm.verify_oids(checker)
except OdmlibOIDError as e:
    print(f"OID error: {e}")

OID error: OID IT.NONEXISTENT referenced in attribute ItemOID is not found.
  Hint: Define an element with OID 'IT.NONEXISTENT' before referencing it via ItemOID.


You can also check for **unreferenced OIDs** — definitions that exist but are never pointed to by any reference. This is useful for finding orphaned metadata.

In [31]:
# Fix the broken reference first so verify_oids passes
mdv.ItemGroupDef.remove(igd2)  # remove the problematic ItemGroupDef

# Add a MethodDef that nothing references
orphan_method = ODM.MethodDef(OID="MT.ORPHAN", Name="Orphaned Method", Type="Computation")
orphan_method.Description = ODM.Description()
orphan_method.Description.TranslatedText.append(
    ODM.TranslatedText(_content="This method is not referenced anywhere", lang="en")
)
mdv.MethodDef.append(orphan_method)

checker = create_oid_checker("define_2_1")
odm.verify_oids(checker)

orphans = odm.unreferenced_oids(checker)
if orphans:
    print("Unreferenced OIDs found:")
    for oid, ref_attr in orphans.items():
        print(f"  {oid} (expected ref attribute: {ref_attr})")
else:
    print("All OIDs are referenced.")

Unreferenced OIDs found:
  MT.ORPHAN (expected ref attribute: MethodOID)


## 7. Conformance Errors (`OdmlibConformanceError`)

odmlib uses [Cerberus](https://docs.python-cerberus.org/) schemas to validate metadata conformance. When conformance validation fails, `OdmlibConformanceError` is raised with a `cerberus_errors` attribute containing the raw Cerberus error dictionary for programmatic inspection.

In [32]:
from odmlib.define_2_1.rules.metadata_schema import MetadataSchema

conformance = MetadataSchema()

# Create an ItemGroupDef with missing required fields for conformance
bad_igd = DEFINE.ItemGroupDef(
    OID="IG.BAD", Name="BAD", Repeating="No",
    IsReferenceData="No", SASDatasetName="BAD",
    Structure="", Purpose="Tabulation",  # empty Structure may fail conformance
)

try:
    bad_igd.verify_conformance(conformance)
    print("Conformance check passed.")
except OdmlibConformanceError as e:
    print(f"Conformance error for: {e.element_type}")
    print(f"Hint: {e.hint}")
    print(f"\nCerberus errors (dict): {e.cerberus_errors}")

Conformance check passed.


The `cerberus_errors` dictionary lets you programmatically iterate over each failing field — useful for generating reports or feeding results into a validation dashboard.

## 8. Collect-All-Errors Mode

By default, odmlib validation stops at the first error (fail-fast). This is useful during development, but when validating a complete document you often want to see **all** the problems at once.

The `validate()` method accepts `collect_errors=True` to accumulate all errors in a single pass. Instead of raising an exception, it returns a list of `OdmlibError` instances.

In [33]:
# First, demonstrate fail-fast mode (the default)
checker = create_oid_checker("define_2_1")
conformance = MetadataSchema()

try:
    odm.validate(collect_errors=False, oid_checker=checker, conformance_checker=conformance)
    print("Validation passed.")
except OdmlibError as e:
    print(f"Fail-fast stopped at first error:")
    print(f"  Type: {type(e).__name__}")
    print(f"  Message: {e}")

Validation passed.


In [34]:
# Now use collect_errors=True to gather all errors
checker = create_oid_checker("define_2_1")

errors = odm.validate(collect_errors=True, oid_checker=checker, conformance_checker=conformance)

if errors:
    print(f"Found {len(errors)} validation error(s):\n")
    for i, err in enumerate(errors, 1):
        print(f"  {i}. [{type(err).__name__}] {err}")
else:
    print("No validation errors found.")

No validation errors found.


### Using `ErrorCollector` Directly

The `validate()` method uses `ErrorCollector` internally, but you can also use it directly to accumulate errors from your own validation logic alongside odmlib's built-in checks.

In [35]:
collector = ErrorCollector()

# Add odmlib errors from validate()
checker = create_oid_checker("define_2_1")
odmlib_errors = odm.validate(collect_errors=True, oid_checker=checker)
for err in odmlib_errors:
    collector.add_error(err)

# Add your own custom validation errors
for igd in mdv.ItemGroupDef:
    if not igd.ItemRef:
        collector.add_error(
            OdmlibValidationError(
                f"ItemGroupDef '{igd.OID}' has no ItemRef children",
                element_type="ItemGroupDef",
                hint="Every dataset should reference at least one variable",
            )
        )

# Check results
print(f"Total errors collected: {len(collector.errors)}")
print(f"Has errors: {collector.has_errors}")

# raise_if_errors() raises a summary exception if there are multiple errors,
# or re-raises the single error if there is exactly one
if collector.has_errors:
    try:
        collector.raise_if_errors()
    except OdmlibValidationError as e:
        print(f"\nSummary exception:\n{e}")

Total errors collected: 0
Has errors: False


## 9. Warnings

odmlib uses Python's `warnings` module for non-fatal issues. The warning hierarchy mirrors the exception hierarchy:

- `OdmlibWarning` — base for all odmlib warnings
- `OdmlibDeprecationWarning` — issued when you use a deprecated feature
- `OdmlibInteroperabilityWarning` — valid constructs that may cause interoperability problems

For example, `reorder_object()` issues an `OdmlibWarning`, and the legacy OIDRef classes issue `OdmlibDeprecationWarning`.

In [36]:
# Capture warnings programmatically
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    # reorder_object issues an OdmlibWarning
    mdv.reorder_object()

print(f"Caught {len(caught)} warning(s):")
for w in caught:
    print(f"  Category: {w.category.__name__}")
    print(f"  Message:  {w.message}")
    print(f"  Is OdmlibWarning: {issubclass(w.category, OdmlibWarning)}")

Caught 1 warning(s):
  Category: OdmlibWarning
  Message:  MetaDataVersion elements are being reordered to match the ODM specification. Use verify_order() before reorder_object() to understand the ordering issue.
  Is OdmlibWarning: True


In [37]:
# Deprecation warning from legacy OIDRef classes
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    # The manual OIDRef classes are deprecated in favor of create_oid_checker()
    from odmlib.define_2_1.rules.oid_ref import OIDRef as LegacyOIDRef
    legacy_checker = LegacyOIDRef()

for w in caught:
    if issubclass(w.category, OdmlibDeprecationWarning):
        print(f"Deprecation warning: {w.message}")
        print(f"\nMigration: Use create_oid_checker('define_2_1') instead")

Deprecation warning: OIDRef is deprecated. Use odmlib.oid_generator.create_oid_checker('define_2_1') instead.

Migration: Use create_oid_checker('define_2_1') instead


## 10. Backward Compatibility

During the v0.2.x transition period, all validation exceptions also inherit from `ValueError` and type exceptions from `TypeError`. This means existing code that catches these built-in types will continue to work.

**Important:** This dual inheritance will be removed in v0.3.0. Update your `except` clauses to use the odmlib-specific types.

In [38]:
# Old code: catching Type Error still works in v0.2.x
try:
    study = ODM.Study(OID=999)
except TypeError:
    print("Caught as TypeError (backward compatible)")

# New code: catching the specific odmlib type
try:
    study = ODM.Study(OID=999)
except OdmlibTypeError as e:
    print(f"Caught as OdmlibTypeError: {e.attribute}")

# Recommended: catch the odmlib base for broad handling
try:
    study = ODM.Study(OID=999)
except OdmlibError as e:
    print(f"Caught as OdmlibError: {type(e).__name__}")

Caught as TypeError (backward compatible)
Caught as OdmlibTypeError: OID
Caught as OdmlibError: OdmlibTypeError


## Summary

The odmlib v0.2.0 error handling improvements give you:

- **Specificity** — catch exactly the error type you care about (`OdmlibOIDError`, `OdmlibTypeError`, etc.)
- **Context** — every exception carries `element_path`, `hint`, `attribute`, and other metadata
- **Collect-all-errors** — `validate(collect_errors=True)` returns every problem in one pass
- **Composability** — `ErrorCollector` lets you mix odmlib errors with your own validation logic
- **Backward compatibility** — existing `except ValueError`/`TypeError` code keeps working through v0.2.x
- **Warnings** — non-fatal issues use Python's standard `warnings` module with odmlib-specific categories